Evaluation & Visualization

This notebook loads the trained SRGAN model and performs comprehensive evaluation:

1. **Load checkpoint** from `training.ipynb`
2. **Qualitative comparison**: LR vs Bicubic vs SRGAN vs HR (side-by-side)
3. **Quantitative metrics**: PSNR and SSIM across the dataset
4. **Loss curves** (loaded from checkpoint)
5. **Zoomed-in detail comparison** to see texture recovery
6. **Per-class analysis** (COVID vs Non-COVID)
7. **Single image inference** demo

In [ ]:
!pip install torch torchvision pillow matplotlib numpy -q

## 1. Imports & Configuration

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from model import CTScanDataset, Generator

plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 120

#  CONFIGURATION

DATASET_PATH    = r"C:\Users\NarisettyDharmaTeja\Downloads\Dataset"
CHECKPOINT_PATH = "checkpoints/srgan_final.pth"      # Path to trained model
BATCH_SIZE      = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Load Trained Model & Dataset

In [ ]:
# Load generator and restore weights
generator = Generator().to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
generator.load_state_dict(checkpoint["generator"])
generator.eval()

history = checkpoint.get("history", {})

print(f"Model loaded from epoch {checkpoint.get('epoch', '?')}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

# Load dataset
dataset = CTScanDataset(DATASET_PATH)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"   Dataset: {len(dataset)} images")

## 3. Training Loss Curves (from checkpoint)
Reload and visualize the loss history saved during training.

In [ ]:
if history:
    n_epochs = len(history["g_loss"])
    epochs_range = range(1, n_epochs + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs_range, history["g_loss"], "o-", color="royalblue", label="Generator")
    axes[0].plot(epochs_range, history["d_loss"], "s-", color="crimson", label="Discriminator")
    axes[0].set_title("G vs D Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history["g_perc"], "o-", color="teal", label="Perceptual")
    axes[1].plot(epochs_range, history["g_adv"],  "s-", color="orange", label="Adversarial")
    axes[1].set_title("Generator Loss Breakdown", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs_range, history["d_real"], "o-", color="green", label="D(real)")
    axes[2].plot(epochs_range, history["d_fake"], "s-", color="red",   label="D(fake)")
    axes[2].axhline(0.693, ls="--", color="gray", alpha=0.5, label="Random (ln2)")
    axes[2].set_title("Discriminator Real vs Fake", fontweight="bold")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("BCE Loss"); axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No training history found in checkpoint.")

## 4. Qualitative Comparison: LR vs Bicubic vs SRGAN vs HR
Side-by-side visual comparison of all four outputs for multiple samples.

In [ ]:
bicubic_up = transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC)

# Helper: tensor → numpy HWC image
def to_img(t):
    return t.cpu().permute(1, 2, 0).clamp(0, 1).numpy()

# Helper: compute PSNR between two tensors
def psnr(pred, target):
    mse = ((pred - target) ** 2).mean().item()
    return 10 * np.log10(1.0 / mse) if mse > 0 else float("inf")

# Get a batch
lr_batch, hr_batch = next(iter(loader))
lr_batch = lr_batch.to(device)

with torch.no_grad():
    sr_batch = generator(lr_batch)

N_SHOW = 6
fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, N_SHOW * 4))
col_titles = ["LR (64*64)", "Bicubic 4*", "SRGAN Output", "HR Ground Truth"]

for i in range(N_SHOW):
    lr_t  = lr_batch[i]
    sr_t  = sr_batch[i]
    hr_t  = hr_batch[i]
    bic_t = bicubic_up(lr_t).clamp(0, 1)

    imgs = [to_img(lr_t), to_img(bic_t), to_img(sr_t), to_img(hr_t)]

    for j, img in enumerate(imgs):
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=12, fontweight="bold")

    # Show PSNR for bicubic and SRGAN
    bic_psnr = psnr(bic_t.cpu(), hr_t)
    sr_psnr  = psnr(sr_t.cpu(), hr_t)
    axes[i, 1].set_xlabel(f"PSNR: {bic_psnr:.2f} dB", fontsize=9)
    axes[i, 2].set_xlabel(f"PSNR: {sr_psnr:.2f} dB", fontsize=9, color="green" if sr_psnr > bic_psnr else "red")

fig.suptitle("LR  →  Bicubic  →  SRGAN  →  Ground Truth", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Quantitative Metrics: PSNR & SSIM
Compute **PSNR** (Peak Signal-to-Noise Ratio) and **SSIM** (Structural Similarity Index) across the full dataset for both Bicubic and SRGAN outputs.

In [ ]:
def ssim(img1, img2, C1=0.01**2, C2=0.03**2):
    """Compute SSIM between two numpy images (H, W, C) in [0, 1]."""
    mu1, mu2 = img1.mean(axis=(0, 1)), img2.mean(axis=(0, 1))
    sigma1_sq = ((img1 - mu1) ** 2).mean(axis=(0, 1))
    sigma2_sq = ((img2 - mu2) ** 2).mean(axis=(0, 1))
    sigma12   = ((img1 - mu1) * (img2 - mu2)).mean(axis=(0, 1))

    num = (2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)
    den = (mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2)
    return (num / den).mean()

# Evaluate over the full dataset
bic_psnrs, sr_psnrs = [], []
bic_ssims, sr_ssims = [], []

print("Evaluating across the full dataset...")
with torch.no_grad():
    for lr, hr in loader:
        lr = lr.to(device)
        sr = generator(lr)

        for i in range(lr.size(0)):
            hr_np  = hr[i].permute(1, 2, 0).numpy()
            sr_np  = sr[i].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
            bic_np = bicubic_up(lr[i]).cpu().permute(1, 2, 0).clamp(0, 1).numpy()

            # PSNR
            bic_psnrs.append(psnr(torch.tensor(bic_np), torch.tensor(hr_np)))
            sr_psnrs.append(psnr(sr[i].cpu(), hr[i]))

            # SSIM
            bic_ssims.append(ssim(bic_np, hr_np))
            sr_ssims.append(ssim(sr_np, hr_np))

print(f"\n{'Metric':<12} {'Bicubic':>12} {'SRGAN':>12} {'Improvement':>14}")
print("─" * 52)
print(f"{'PSNR (dB)':<12} {np.mean(bic_psnrs):>12.2f} {np.mean(sr_psnrs):>12.2f} {np.mean(sr_psnrs) - np.mean(bic_psnrs):>+12.2f} dB")
print(f"{'SSIM':<12} {np.mean(bic_ssims):>12.4f} {np.mean(sr_ssims):>12.4f} {np.mean(sr_ssims) - np.mean(bic_ssims):>+12.4f}")

## 6. Metrics Distribution
Histograms of PSNR and SSIM values across all images — Bicubic vs SRGAN.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PSNR histogram
axes[0].hist(bic_psnrs, bins=40, alpha=0.6, color="orange", edgecolor="black", label=f"Bicubic (μ={np.mean(bic_psnrs):.2f})")
axes[0].hist(sr_psnrs,  bins=40, alpha=0.6, color="royalblue", edgecolor="black", label=f"SRGAN (μ={np.mean(sr_psnrs):.2f})")
axes[0].set_title("PSNR Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("PSNR (dB)")
axes[0].set_ylabel("Count")
axes[0].legend()

# SSIM histogram
axes[1].hist(bic_ssims, bins=40, alpha=0.6, color="orange", edgecolor="black", label=f"Bicubic (μ={np.mean(bic_ssims):.4f})")
axes[1].hist(sr_ssims,  bins=40, alpha=0.6, color="royalblue", edgecolor="black", label=f"SRGAN (μ={np.mean(sr_ssims):.4f})")
axes[1].set_title("SSIM Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("SSIM")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Zoomed-In Detail Comparison
Crop and zoom into a region to visualize texture recovery by SRGAN vs Bicubic.

In [ ]:
# Pick 3 samples and zoom into a 80*80 crop from the centre
crop_size = 80
start = (256 - crop_size) // 2

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
col_titles = ["Bicubic (full)", "Bicubic (zoom)", "SRGAN (zoom)", "HR Ground Truth (zoom)"]

for i in range(3):
    lr_t = lr_batch[i]
    sr_t = sr_batch[i]
    hr_t = hr_batch[i]
    bic_t = bicubic_up(lr_t).clamp(0, 1)

    bic_np = to_img(bic_t)
    sr_np  = to_img(sr_t)
    hr_np  = hr_t.permute(1, 2, 0).clamp(0, 1).numpy()

    # Crop centre region
    s, e = start, start + crop_size
    bic_crop = bic_np[s:e, s:e]
    sr_crop  = sr_np[s:e, s:e]
    hr_crop  = hr_np[s:e, s:e]

    axes[i, 0].imshow(bic_np)
    # Draw crop rectangle
    rect = plt.Rectangle((s, s), crop_size, crop_size, linewidth=2, edgecolor="lime", facecolor="none")
    axes[i, 0].add_patch(rect)

    axes[i, 1].imshow(bic_crop)
    axes[i, 2].imshow(sr_crop)
    axes[i, 3].imshow(hr_crop)

    for j in range(4):
        axes[i, j].axis("off")
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=11, fontweight="bold")

fig.suptitle("Zoomed-In Detail Comparison (centre 80*80 crop)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Error Heatmaps
Pixel-wise absolute error maps between SRGAN output and ground truth.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
col_titles = ["SRGAN Output", "HR Ground Truth", "Absolute Error (SRGAN)", "Absolute Error (Bicubic)"]

for i in range(3):
    sr_np  = to_img(sr_batch[i])
    hr_np  = hr_batch[i].permute(1, 2, 0).clamp(0, 1).numpy()
    bic_np = to_img(bicubic_up(lr_batch[i]).clamp(0, 1))

    sr_err  = np.abs(sr_np - hr_np).mean(axis=2)   # Mean across RGB channels
    bic_err = np.abs(bic_np - hr_np).mean(axis=2)

    axes[i, 0].imshow(sr_np)
    axes[i, 1].imshow(hr_np)
    im_sr  = axes[i, 2].imshow(sr_err,  cmap="hot", vmin=0, vmax=0.3)
    im_bic = axes[i, 3].imshow(bic_err, cmap="hot", vmin=0, vmax=0.3)

    for j in range(4):
        axes[i, j].axis("off")
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=11, fontweight="bold")

    axes[i, 2].set_xlabel(f"MAE: {sr_err.mean():.4f}", fontsize=9)
    axes[i, 3].set_xlabel(f"MAE: {bic_err.mean():.4f}", fontsize=9)

fig.colorbar(im_sr, ax=axes[:, 2].tolist(), shrink=0.6, label="Abs Error")
fig.colorbar(im_bic, ax=axes[:, 3].tolist(), shrink=0.6, label="Abs Error")
fig.suptitle("Pixel-wise Error Heatmaps", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Single Image Inference Demo
Load any image, downscale it, and run SRGAN super-resolution.

In [ ]:
def super_resolve(image_path, generator, device):
    """Run SRGAN on a single image file.

    Parameters
    ----------
    image_path : str   - Path to any image file
    generator  : nn.Module - Trained SRGAN generator
    device     : torch.device

    Returns
    -------
    lr_img, sr_img, hr_img : numpy arrays (H, W, 3) in [0, 1]
    """
    img = Image.open(image_path).convert("RGB")
    hr = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])(img)
    lr = transforms.Compose([
        transforms.Resize((64, 64), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor()
    ])(img)

    with torch.no_grad():
        sr = generator(lr.unsqueeze(0).to(device)).squeeze(0)

    return (
        lr.permute(1, 2, 0).numpy(),
        sr.cpu().permute(1, 2, 0).clamp(0, 1).numpy(),
        hr.permute(1, 2, 0).numpy(),
    )

# Demo: pick a random image from the dataset
import random
demo_path = random.choice(dataset.images)

lr_img, sr_img, hr_img = super_resolve(demo_path, generator, device)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [lr_img, sr_img, hr_img],
                           ["LR Input (64×64)", "SRGAN Output (256×256)", "HR Ground Truth (256×256)"]):
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis("off")

plt.suptitle(f"Single Image Demo: {os.path.basename(demo_path)}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

sr_psnr_val = psnr(torch.tensor(sr_img), torch.tensor(hr_img))
sr_ssim_val = ssim(sr_img, hr_img)
print(f"PSNR: {sr_psnr_val:.2f} dB  |  SSIM: {sr_ssim_val:.4f}")

## 10. Frequency Domain Analysis
Compare the frequency content of Bicubic vs SRGAN vs HR using FFT magnitude spectra.
SRGAN should recover more high-frequency detail than bicubic upscaling.

In [ ]:
def fft_magnitude(img_np):
    """Compute log-magnitude FFT spectrum of a grayscale image."""
    gray = np.mean(img_np, axis=2)
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    return np.log1p(np.abs(fshift))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles = ["Bicubic 4*", "SRGAN Output", "HR Ground Truth"]

for i in range(2):
    sr_np  = to_img(sr_batch[i])
    hr_np  = hr_batch[i].permute(1, 2, 0).clamp(0, 1).numpy()
    bic_np = to_img(bicubic_up(lr_batch[i]).clamp(0, 1))

    for j, (img, title) in enumerate(zip([bic_np, sr_np, hr_np], titles)):
        spectrum = fft_magnitude(img)
        axes[i, j].imshow(spectrum, cmap="inferno")
        axes[i, j].axis("off")
        if i == 0:
            axes[i, j].set_title(title, fontsize=12, fontweight="bold")

fig.suptitle("FFT Magnitude Spectra (higher = more high-freq detail)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()